# IDA CFG Analysis with ipysigma

This notebook demonstrates how to extract, load, and visualize the Control Flow Graph (CFG) from an IDA database.


### 1. Extract CFG from IDA Database
This step requires `idapro` 9.0+ and a valid license.


In [4]:
import sys
import os
from src.extract_cfg import extract_cfg_from_db

# Replace with the path to your IDA database (.idb or .i64)
db_path = "/Users/mark/windows_share/test/reorder_and_pad.exe.i64"
json_path = "../data/parsed_xref_graph.json"

if os.path.exists(db_path):
    json_path = extract_cfg_from_db(db_path, output_path=json_path)
else:
    print(f"Database {db_path} not found. Skipping extraction.")


Opening database: /Users/mark/windows_share/test/reorder_and_pad.exe.i64
Number of functions found: 343
Extracting imported functions...
Found 77 imported functions.
Extracting functions and intra-function edges...
Found 343 functions.
Extracting inter-function calls...
Found non-call link to main 5368713352 from sub_140001000 @ 0x140001000
Extracting calls to imported functions...
CFG exported to reorder_and_pad2.json


### 2. Load and Visualize the Graph with ipysigma


In [5]:
from idadex import idaapi
from typing import Hashable
from networkx import k_core, DiGraph
import importlib
importlib.invalidate_caches()
from ipysigma import Sigma
import pandas as pd
import networkx as nx
import src.visualize_cfg as cfg
importlib.reload(cfg)
import os




def prune_graph(og: DiGraph[Hashable]) -> DiGraph[Hashable]:
    to_return = og.copy()
    print(f"Graph loaded: {len(to_return.nodes)} nodes, {len(to_return.edges)} edges")
    entrypoint = [x for x in to_return.nodes() if to_return.nodes[x]['entry_point'] == True]
    print(f"entrypoint is {entrypoint}")

    # remove self loops
    to_return.remove_edges_from(nx.selfloop_edges(to_return))

    to_return = collapse_thunks(to_return)
    to_return = cfg.collapse_chains(to_return)

    # remove nodes with degree <= 1
    to_return = k_core(to_return, 2)
    entrypoint = [x for x in to_return.nodes() if to_return.nodes[x]['entry_point'] == True]
    print(f"entrypoint is {entrypoint}")
    # to_be_removed = [x for  x in G.nodes() if G.degree()[x] <= 1]
    # print(f"Number of nodes to be removed: {len(to_be_removed)}")
    # G.remove_nodes_from(to_be_removed)
    # # Basic info
    # print(f"Number of functions after removing degree <= 1: {len(G.nodes)}")
    #
    # print(f"Number of nodes after collapsing chains: {len(G.nodes)}")
    # entrypoint = [x for x in G.nodes() if G.nodes[x]['entry_point'] == True]
    # if not entrypoint:
    #     print("No entrypoint found. Please check the graph.")
    #     raise ValueError("No entrypoint found")
    print(f"Graph pruned: {len(to_return.nodes)} nodes, {len(to_return.edges)} edges")
    return to_return

def collapse_thunks(G: nx.DiGraph):
    collapsed_G = G.copy()
    nodes_to_process = list(collapsed_G.nodes())

    for node in nodes_to_process:

        flags = collapsed_G.nodes[node].get('flags', {})
        if flags & idaapi.FUNC_THUNK:
            preds = list(collapsed_G.predecessors(node))
            succs = list(collapsed_G.successors(node))
            if len(succs) == 1:
                collapsed_G.remove_node(node)
                for pred in preds:
                    collapsed_G.add_edge(pred, succs[0])
    return collapsed_G

# Load the graph data
G = cfg.load_cfg(json_path)
if G is None:
    print(f"Could not load graph from {json_path}. Please make sure the file exists and it is a valid CFG JSON.")
    exit()

assert(G is not None)
pruned = prune_graph(G)

largest_degree = 0
def set_size(n, d) -> int:
    return len(d.get('instrs'))+G.out_degree(n)

Graph loaded: 5174 nodes, 8729 edges
entrypoint is ['0x1400155d8']
entrypoint is ['0x1400155d8']
Graph pruned: 4881 nodes, 8345 edges


In [6]:

# Visualize with ipysigma
widget = Sigma(
    pruned,
    node_label='label',
    raw_node_color=lambda n, d: "red" if "main" in d.get("func") else d.get('color'),
    node_size=set_size,
    edge_color='type',
    label_font='sans-serif',
    default_edge_type='arrow',
    start_layout=10,
)
print(largest_degree)
display(widget)


0


Sigma(nx.DiGraph with 4,881 nodes and 8,345 edges)

Ways to reduce:
- strongly connected components
- remove nodes with degree <= 1
- clique detection and collapse
- graph community collapse
min cuts
- graph spectral?
- laplacian?
- adamic-adar index?

In [ ]:
## needed to reload code changed on disk, this is the correct import
import src.summarize_graph as sg
importlib.reload(sg)
summaries = sg.summarize_graph(pruned, base_url="http://192.168.1.101:8000/v1", max_concurrent=256, model="qwen3-coder-next", cache_path="../data/cache_full_func.json")


In [ ]:
for node in pruned.nodes():
    pruned.nodes[node]['summary'] = summaries.get(pruned.nodes[node].get('func'))


In [ ]:
# visualize with ipysigma, now with summaries
widget = Sigma(
    pruned,
    raw_node_color=lambda n, d: "red" if "main" in d.get    ("func") else d.get('color'),
    node_size=set_size,
    edge_color='type',
    label_font='sans-serif',
    default_edge_type='arrow',
    start_layout=10,
)
display(widget)


### 3. Extract Subset: windows::main and touching edges


In [7]:
# Identify nodes from windows::main
main_nodes = [n for n, d in pruned.nodes(data=True) if d.get('func') and "windows::main" in d.get('func')]
print(f"Found {len(main_nodes)} nodes in windows::main.")

# Get all edges touching these nodes
touching_edges = []
for u, v in pruned.edges():
    if u in main_nodes or v in main_nodes:
        touching_edges.append((u, v))

# Create the subset graph
nodes_in_subset = set(main_nodes)
for u, v in touching_edges:
    nodes_in_subset.add(u)
    nodes_in_subset.add(v)

main_subset = pruned.subgraph(nodes_in_subset).copy()

# Keep only edges that touch a windows::main block
edges_to_remove = []
for u, v in main_subset.edges():
    if u not in main_nodes and v not in main_nodes:
        edges_to_remove.append((u, v))
main_subset.remove_edges_from(edges_to_remove)

print(f"Subset graph: {len(main_subset.nodes)} nodes, {len(main_subset.edges)} edges")

# Visualize the subset
widget_subset = Sigma(
    main_subset,
    node_label='label',
    raw_node_color=lambda n, d: "red" if n in main_nodes else "blue",
    node_size=set_size,
    edge_color='type',
    label_font='sans-serif',
    default_edge_type='arrow',
    start_layout=10,
)
display(widget_subset)

# # Save subset to JSON if needed
# import json
# from networkx.readwrite import json_graph
# subset_data = json_graph.node_link_data(main_subset)
# with open("main_subset.json", "w") as f:
#     json.dump(subset_data, f, indent=2)
# print("Saved subset to main_subset.json")

Found 6 nodes in windows::main.
Subset graph: 12 nodes, 14 edges


Sigma(nx.DiGraph with 12 nodes and 14 edges)